# M11 economic resolution — independent recomputation

**Objective.** Authenticate the sealed M11 central evidence and independently recompute every ledger, saving, bootstrap interval, gate, and terminal economic disposition.

**Pass condition.** Exact agreement without importing YieldForge production aggregation or decision functions.


In [ ]:
from __future__ import annotations
import gzip, hashlib, io, json, os, stat
from decimal import Decimal, ROUND_HALF_EVEN, localcontext
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import HTML, display

Q12 = Decimal("0.000000000001")
MAX_JSON = MAX_GZIP = 4 * 1024 * 1024
MAX_RAW = 128 * 1024 * 1024
ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "experiments/results").is_dir())
RESULTS = ROOT / "experiments/results/m11-economic-resolution"

def duplicate_keys(pairs):
    out = {}
    for key, value in pairs:
        if key in out: raise ValueError(f"duplicate JSON key: {key}")
        out[key] = value
    return out

def reject_nonfinite(token): raise ValueError(f"non-finite JSON constant: {token}")
def fingerprint(info): return (info.st_dev, info.st_ino, info.st_size, info.st_mtime_ns, info.st_ctime_ns)

def bounded_read(path, cap):
    path = Path(path); before = path.lstat()
    if stat.S_ISLNK(before.st_mode) or not stat.S_ISREG(before.st_mode) or before.st_size > cap:
        raise ValueError(f"unsafe evidence file: {path.name}")
    fd = os.open(path, os.O_RDONLY | getattr(os, "O_NOFOLLOW", 0))
    try:
        if fingerprint(os.fstat(fd)) != fingerprint(before): raise ValueError("file changed before open")
        chunks, total = [], 0
        while True:
            chunk = os.read(fd, min(1024 * 1024, cap + 1 - total))
            if not chunk: break
            chunks.append(chunk); total += len(chunk)
            if total > cap: raise ValueError("file exceeded bound")
    finally: os.close(fd)
    if fingerprint(path.lstat()) != fingerprint(before): raise ValueError("file changed during read")
    return b"".join(chunks)

def parse_json(raw):
    return json.loads(raw.decode("utf-8", errors="strict"), object_pairs_hook=duplicate_keys, parse_constant=reject_nonfinite)
def compact(value): return json.dumps(value, allow_nan=False, separators=(",", ":"), sort_keys=True).encode()
def pretty(value): return (json.dumps(value, allow_nan=False, indent=2, sort_keys=True) + "\n").encode()
def digest(raw): return hashlib.sha256(raw).hexdigest()

def verify_identity(value, id_field, prefix):
    semantic = dict(value); semantic.pop(id_field); semantic.pop("content_sha256")
    sha = digest(compact(semantic))
    assert value[id_field] == prefix + sha[:24] and value["content_sha256"] == "sha256:" + sha
    return sha

def load_plain(filename, id_field, prefix):
    raw = bounded_read(RESULTS / filename, MAX_JSON); value = parse_json(raw)
    assert raw == pretty(value); verify_identity(value, id_field, prefix)
    return value

def load_sidecar(filename):
    compressed = bounded_read(RESULTS / filename, MAX_GZIP)
    assert compressed[:4] == b"\x1f\x8b\x08\x00" and compressed[4:8] == b"\0" * 4
    chunks, total = [], 0
    with gzip.GzipFile(fileobj=io.BytesIO(compressed)) as handle:
        while True:
            chunk = handle.read(min(1024 * 1024, MAX_RAW + 1 - total))
            if not chunk: break
            chunks.append(chunk); total += len(chunk)
            if total > MAX_RAW: raise ValueError("inflated sidecar exceeded bound")
    return compressed, b"".join(chunks)

def metric12(value):
    with localcontext() as ctx:
        ctx.prec = 50; value = value if isinstance(value, Decimal) else Decimal(str(value))
        if not value.is_finite(): raise ValueError("metric must be finite")
        value = value.quantize(Q12, rounding=ROUND_HALF_EVEN)
    if value == 0: value = Decimal(0)
    return format(value, ".12f")

print(f"Independent audit environment ready: NumPy {np.__version__}, Decimal precision 50.")


## 1. Pin and authenticate the terminal artifacts

Selection is outcome-independent: exact filenames, IDs, and hashes are fixed below. The readers reject symlinks, changing or oversized files, duplicate keys, NaN/Infinity, non-canonical pretty JSON, and semantic identity mismatches.


In [ ]:
MANIFEST_FILE = "m11-economic-central-manifest-71171ff1cb601f546f55b78eda8dc2b81d60d7e02949042a55d53feb29e5dcf2.json"
MANIFEST_PIN = ("yfm11econcentral-71171ff1cb601f546f55b78e", "sha256:71171ff1cb601f546f55b78eda8dc2b81d60d7e02949042a55d53feb29e5dcf2")
SEGMENT_PINS = {
    "loco-2dics": ("m11-economic-central-segment-loco-2dics-0474c6399f3465fe3f50ad455c80a133ec967febedfe7f66bbad373080cd212e.json", "yfm11econsegsummary-0474c6399f3465fe3f50ad45", "sha256:0474c6399f3465fe3f50ad455c80a133ec967febedfe7f66bbad373080cd212e"),
    "lectra-m3-m4": ("m11-economic-central-segment-lectra-m3-m4-746e83d7bdc4482ec2276adccf2646b611b4d2267a4d6b3c201cff85e403ef9f.json", "yfm11econsegsummary-746e83d7bdc4482ec2276adc", "sha256:746e83d7bdc4482ec2276adccf2646b611b4d2267a4d6b3c201cff85e403ef9f"),
}
manifest = load_plain(MANIFEST_FILE, "manifest_id", "yfm11econcentral-")
assert (manifest["manifest_id"], manifest["content_sha256"]) == MANIFEST_PIN
summaries = {}
for corpus, (filename, expected_id, expected_sha) in SEGMENT_PINS.items():
    summary = load_plain(filename, "summary_id", "yfm11econsegsummary-")
    assert summary["corpus_id"] == corpus and (summary["summary_id"], summary["content_sha256"]) == (expected_id, expected_sha)
    summaries[corpus] = summary
assert manifest["segment_summaries"] == [summaries["loco-2dics"], summaries["lectra-m3-m4"]]
print("Pinned manifest and both standalone segment summaries authenticate exactly.")


## 2. Reconcile all 40 checkpoints and sidecars

The embedded manifest checkpoints are compared byte-for-byte semantically with the standalone checkpoint files. Each compact receipt authenticates one compressed cell; compressed bytes, gzip header, uncompressed bytes, canonical cell JSON, and cell semantic identity are all checked.


In [ ]:
checkpoints = manifest["checkpoints"]
assert len(checkpoints) == manifest["total_cell_count"] == 40
assert manifest["checkpoint_ids"] == [x["checkpoint_id"] for x in checkpoints]
assert manifest["checkpoint_content_sha256s"] == [x["content_sha256"] for x in checkpoints]
assert [x["execution_position"] for x in checkpoints] == list(range(40))
assert [x["corpus_id"] for x in checkpoints] == ["loco-2dics"] * 20 + ["lectra-m3-m4"] * 20
checkpoint_files, sidecar_files, receipts, cells = set(), set(), [], []
for position, embedded in enumerate(checkpoints):
    sha = embedded["content_sha256"].removeprefix("sha256:")
    filename = f"m11-economic-central-cell-checkpoint-{position:02d}-{sha}.json"; checkpoint_files.add(filename)
    assert load_plain(filename, "checkpoint_id", "yfm11econcellcp-") == embedded
    assert embedded["complete"] is True and embedded["prior_failure_attempt_count"] == 0
    receipt = embedded["receipt"]; verify_identity(receipt, "receipt_id", "yfm11g3cellrcpt-")
    assert (embedded["receipt_id"], embedded["receipt_content_sha256"]) == (receipt["receipt_id"], receipt["content_sha256"])
    assert receipt["compression"] == "gzip-level-6-mtime-0-flags-0"
    expected_sidecar = f"m11-gate3-central-cell-{receipt['cell_content_sha256'].removeprefix('sha256:')}-{receipt['compressed_raw_sha256'].removeprefix('sha256:')}.json.gz"
    assert receipt["sidecar_name"] == expected_sidecar
    sidecar_files.add(receipt["sidecar_name"]); compressed, raw = load_sidecar(receipt["sidecar_name"])
    assert len(compressed) == receipt["compressed_byte_count"] and "sha256:" + digest(compressed) == receipt["compressed_raw_sha256"]
    assert len(raw) == receipt["uncompressed_byte_count"]
    cell = parse_json(raw); assert raw == pretty(cell); verify_identity(cell, "cell_id", "yfm11g3cell-")
    assert (cell["cell_id"], cell["content_sha256"], cell["corpus_id"], cell["stream_id"]) == (receipt["cell_id"], receipt["cell_content_sha256"], receipt["corpus_id"], receipt["stream_id"])
    receipts.append(receipt); cells.append(cell)
assert {p.name for p in RESULTS.glob("m11-economic-central-cell-checkpoint-*.json")} == checkpoint_files
assert {p.name for p in RESULTS.glob("m11-gate3-central-cell-*.json.gz")} == sidecar_files
assert not tuple(RESULTS.glob("m11-economic-central-cell-failure-*.json"))
for start, corpus in ((0, "loco-2dics"), (20, "lectra-m3-m4")):
    block, summary = receipts[start:start + 20], summaries[corpus]
    assert summary["receipts"] == block
    assert summary["canonical_stream_ids"] == [x["stream_id"] for x in block] and len(set(summary["canonical_stream_ids"])) == 20
    assert summary["receipt_ids"] == [x["receipt_id"] for x in block] and summary["receipt_content_sha256s"] == [x["content_sha256"] for x in block]
    assert summary["cell_ids"] == [x["cell_id"] for x in block] and summary["cell_content_sha256s"] == [x["cell_content_sha256"] for x in block]
pd.DataFrame([
    {"evidence": "checkpoints", "expected": 40, "verified": len(checkpoint_files)},
    {"evidence": "gzip sidecars", "expected": 40, "verified": len(sidecar_files)},
    {"evidence": "compact receipts", "expected": 40, "verified": len(receipts)},
    {"evidence": "failure receipts", "expected": 0, "verified": 0},
])


## 3. Recompute accounting and per-stream economics

For each of 120 arm ledgers: `net = purchase + storage + return + retrieval − scrap − terminal`. For each paired stream: `F = 100(B−F)/B`, `K = 100(B−K)/B`, and `headroom = 100(K−F)/B`, using precision 50 and twelve-place half-even rounding.


In [ ]:
ARMS = (("baseline", "baseline_costs", "baseline_cost"), ("full_future", "full_future_costs", "full_future_cost"), ("known_only", "known_only_costs", "known_only_cost"))
ledger_count, stream_rows, component_rows = 0, [], []
for receipt, cell in zip(receipts, cells, strict=True):
    for arm, ledger_key, cost_key in ARMS:
        ledger = receipt[ledger_key]; verify_identity(ledger, "ledger_id", "yfm11g3led-")
        with localcontext() as ctx:
            ctx.prec = 50
            net = Decimal(ledger["purchase_cost"]) + Decimal(ledger["storage_cost"]) + Decimal(ledger["return_handling_cost"]) + Decimal(ledger["retrieval_handling_cost"]) - Decimal(ledger["scrap_proceeds"]) - Decimal(ledger["terminal_credit"])
        assert net == Decimal(ledger["net_cost"]) == Decimal(receipt[cost_key]) and cell[arm]["final_costs"] == ledger and cell[cost_key] == receipt[cost_key]
        ledger_count += 1
        component_rows.append({"segment": receipt["corpus_id"], "arm": arm, **{k: Decimal(ledger[k]) for k in ("purchase_cost", "storage_cost", "return_handling_cost", "retrieval_handling_cost", "scrap_proceeds", "terminal_credit", "net_cost")}})
    with localcontext() as ctx:
        ctx.prec = 50; B, F, K = map(Decimal, (receipt["baseline_cost"], receipt["full_future_cost"], receipt["known_only_cost"])); assert B > 0
        f = metric12(Decimal(100) * (B - F) / B); k = metric12(Decimal(100) * (B - K) / B); h = metric12(Decimal(100) * (K - F) / B)
    assert f == receipt["full_future_savings_percent"] == cell["full_future_savings_percent"]
    assert k == receipt["known_only_causal_savings_percent"]
    assert h == receipt["unknown_future_contribution_points"] == cell["unknown_future_contribution_points"]
    stream_rows.append({"segment": receipt["corpus_id"], "stream_id": receipt["stream_id"], "B": B, "F": F, "K": K, "f": Decimal(f), "k": Decimal(k), "h": Decimal(h)})
assert ledger_count == 120 and len(stream_rows) == 40
component_means = pd.DataFrame(component_rows).groupby(["segment", "arm"], sort=False).mean()
print(f"Reconciled {ledger_count} exact ledgers and {len(stream_rows)} paired stream metrics.")
component_means


## 4. Reproduce statistics and frozen decisions

The executor ran LOCo first, but the statistical contract consumes PCG64(0) draws in **Lectra-then-LOCo** order. Confidence intervals use 10,000 paired-stream resamples and NumPy's linear quantile (type 7).


In [ ]:
rng = np.random.Generator(np.random.PCG64(0))
indices = {"lectra-m3-m4": rng.integers(0, 20, (10_000, 20), dtype=np.int64), "loco-2dics": rng.integers(0, 20, (10_000, 20), dtype=np.int64)}
def median20(values):
    values = sorted(values); return (values[9] + values[10]) / Decimal(2)
def statistics_for(corpus):
    rows = [r for r in stream_rows if r["segment"] == corpus]; assert len(rows) == 20
    fv, kv, hv = ([r[key] for r in rows] for key in ("f", "k", "h"))
    with localcontext() as ctx:
        ctx.prec = 50
        fm, km, hm = (sum(v, Decimal(0)) / Decimal(20) for v in (fv, kv, hv))
        ff, kf = Decimal(sum(v > 0 for v in fv)) / Decimal(20), Decimal(sum(v > 0 for v in kv)) / Decimal(20)
    fa, ka = np.asarray([float(v) for v in fv]), np.asarray([float(v) for v in kv])
    fb = np.quantile(fa[indices[corpus]].mean(1), (.025, .975), method="linear"); kb = np.quantile(ka[indices[corpus]].mean(1), (.025, .975), method="linear")
    return {"f_mean_savings_percent": metric12(fm), "f_mean_ci_lower_percent": metric12(float(fb[0])), "f_mean_ci_upper_percent": metric12(float(fb[1])), "f_median_savings_percent": metric12(median20(fv)), "f_positive_stream_count": sum(v > 0 for v in fv), "f_positive_stream_fraction": metric12(ff), "unknown_headroom_mean_percentage_points": metric12(hm), "k_mean_savings_percent": metric12(km), "k_mean_ci_lower_percent": metric12(float(kb[0])), "k_mean_ci_upper_percent": metric12(float(kb[1])), "k_median_savings_percent": metric12(median20(kv)), "k_positive_stream_count": sum(v > 0 for v in kv), "k_positive_stream_fraction": metric12(kf)}
FIELDS = ("f_mean_savings_percent", "f_mean_ci_lower_percent", "f_mean_ci_upper_percent", "f_median_savings_percent", "f_positive_stream_count", "f_positive_stream_fraction", "unknown_headroom_mean_percentage_points", "k_mean_savings_percent", "k_mean_ci_lower_percent", "k_mean_ci_upper_percent", "k_median_savings_percent", "k_positive_stream_count", "k_positive_stream_fraction")
metrics = {}
excluded = {"summary_id", "content_sha256", "source_metrics_id", "source_metrics_content_sha256", "decision", "next_action", "complete", "productization_authorized", "bounded_pilot_authorized"}
for corpus, summary in summaries.items():
    metrics[corpus] = statistics_for(corpus); assert {k: summary[k] for k in FIELDS} == metrics[corpus]
    source_sha = digest(compact({k: v for k, v in summary.items() if k not in excluded}))
    assert summary["source_metrics_id"] == "yfm11econmetrics-" + source_sha[:24] and summary["source_metrics_content_sha256"] == "sha256:" + source_sha

def classify(m):
    ff = {"f_mean_passes": Decimal(m["f_mean_savings_percent"]) >= Decimal("2.5"), "f_lcb_passes": Decimal(m["f_mean_ci_lower_percent"]) > 0, "f_median_passes": Decimal(m["f_median_savings_percent"]) > 0, "f_positive_fraction_passes": Decimal(m["f_positive_stream_fraction"]) > Decimal(".5")}
    kf = {"k_mean_passes": Decimal(m["k_mean_savings_percent"]) >= Decimal("1.5"), "k_lcb_passes": Decimal(m["k_mean_ci_lower_percent"]) > 0, "k_median_passes": Decimal(m["k_median_savings_percent"]) > 0, "k_positive_fraction_passes": Decimal(m["k_positive_stream_fraction"]) > Decimal(".5")}
    fg, kg = all(ff.values()), all(kf.values()); candidate = "causal_candidate" if kg else "forecast_candidate" if fg else "current_segment_red"
    return {**ff, "f_economic_green": fg, "unknown_headroom_diagnostic_green": Decimal(m["unknown_headroom_mean_percentage_points"]) >= Decimal("1.5"), **kf, "k_causal_green": kg, "candidate_classification": candidate}
classes = {}
for corpus, summary in summaries.items():
    expected, decision = classify(metrics[corpus]), summary["decision"]; verify_identity(decision, "decision_id", "yfm11econseg-")
    assert {k: decision[k] for k in expected} == expected and decision["source_summary_id"] == summary["source_metrics_id"] and decision["source_summary_content_sha256"] == summary["source_metrics_content_sha256"]
    candidate = expected["candidate_classification"]
    if corpus == "loco-2dics": expected_loco = expected_next = {"causal_candidate": "CONTINUE_ADVERSE_LOCO", "forecast_candidate": "CONTINUE_FORECAST_LOCO", "current_segment_red": "CONTINUE_LECTRA_SCREEN"}[candidate]
    else: expected_loco, expected_next = None, {"causal_candidate": "CONTINUE_ADVERSE_LECTRA", "forecast_candidate": "CONTINUE_FORECAST_LECTRA", "current_segment_red": None}[candidate]
    assert summary["next_action"] == expected_next and decision["loco_next_step"] == expected_loco
    classes[corpus] = expected["candidate_classification"]
ordered = ("loco-2dics", "lectra-m3-m4"); causal = [x for x in ordered if classes[x] == "causal_candidate"]; forecast = [x for x in ordered if classes[x] == "forecast_candidate"]
if causal: disposition, actions, terminal = "CONTINUE_ADVERSE_SEGMENT_CONFIRMATION", ["CONTINUE_ADVERSE_LOCO" if x == "loco-2dics" else "CONTINUE_ADVERSE_LECTRA" for x in causal], False
elif forecast: disposition, actions, terminal = "CONTINUE_FORECAST_SEGMENT_CONFIRMATION", ["CONTINUE_FORECAST_LOCO" if x == "loco-2dics" else "CONTINUE_FORECAST_LECTRA" for x in forecast], False
else: disposition, actions, terminal = "INSUFFICIENT_CURRENT_MODELED_VALUE", [], True
cross = manifest["cross_segment_decision"]; verify_identity(cross, "decision_id", "yfm11econglobal-")
assert cross["segment_decisions"] == [summaries[x]["decision"] for x in ordered] and (cross["global_disposition"], cross["next_actions"], cross["terminal"]) == (disposition, actions, terminal)
assert (manifest["global_disposition"], manifest["next_actions"], manifest["economic_value_resolved"]) == (disposition, actions, terminal)
assert manifest["status"] == "insufficient_current_modeled_value" and terminal is True and manifest["productization_authorized"] is manifest["bounded_pilot_authorized"] is False
print(f"Bootstrap, thresholds, flags, and reducer match exactly: {disposition}")


## Results

F is the perfect-future-information arm; K is the deployable known-only causal arm. An arm is green only when magnitude, positive lower bound, positive median, and savings on more than half the streams all pass.


In [ ]:
rows = []
for corpus in ("loco-2dics", "lectra-m3-m4"):
    m, d = metrics[corpus], summaries[corpus]["decision"]
    rows.append({"segment": corpus, "F mean %": m["f_mean_savings_percent"], "F 95% CI": f"[{m['f_mean_ci_lower_percent']}, {m['f_mean_ci_upper_percent']}]", "F median %": m["f_median_savings_percent"], "F positive": f"{m['f_positive_stream_count']}/20", "F gate": "PASS" if d["f_economic_green"] else "FAIL", "K mean %": m["k_mean_savings_percent"], "K 95% CI": f"[{m['k_mean_ci_lower_percent']}, {m['k_mean_ci_upper_percent']}]", "K median %": m["k_median_savings_percent"], "K positive": f"{m['k_positive_stream_count']}/20", "K gate": "PASS" if d["k_causal_green"] else "FAIL", "classification": classes[corpus]})
display(pd.DataFrame(rows).set_index("segment"))

def bar(label, value, threshold, color):
    width, marker = max(0, min(100, float(value / Decimal("3") * 100))), float(threshold / Decimal("3") * 100)
    return f"<div style='margin:6px'><span style='display:inline-block;width:180px'>{label}: <b>{value:.3f}%</b></span><span style='display:inline-block;position:relative;width:400px;height:17px;background:#edf0f4'><span style='display:block;width:{width:.3f}%;height:17px;background:{color}'></span><span style='position:absolute;left:{marker:.3f}%;top:-2px;height:21px;border-left:2px solid #222'></span></span></div>"
chart = ["<b>Mean savings vs arm-specific threshold marker</b>"]
for corpus in ("loco-2dics", "lectra-m3-m4"):
    chart += [bar(corpus + " F", Decimal(metrics[corpus]["f_mean_savings_percent"]), Decimal("2.5"), "#8c6bb1"), bar(corpus + " K", Decimal(metrics[corpus]["k_mean_savings_percent"]), Decimal("1.5"), "#2b8cbe")]
display(HTML("<div style='font-family:system-ui'>" + "".join(chart) + "<small>Black marker = frozen mean threshold; reliability conditions are in the table.</small></div>"))
AUDIT_RESULT = {"status": "verified", "manifest_id": manifest["manifest_id"], "manifest_content_sha256": manifest["content_sha256"], "verified_checkpoints": len(checkpoints), "verified_sidecars": len(sidecar_files), "reconciled_ledgers": ledger_count, "segments": {c: {"f_mean_savings_percent": metrics[c]["f_mean_savings_percent"], "k_mean_savings_percent": metrics[c]["k_mean_savings_percent"], "classification": classes[c]} for c in ordered}, "global_disposition": disposition, "economic_value_resolved": terminal, "productization_authorized": False, "bounded_pilot_authorized": False}
print(json.dumps(AUDIT_RESULT, indent=2, sort_keys=True))


## Provenance and claim ceiling

- **source_observed:** geometry references and source demand.
- **externally_anchored:** Lectra candidate references.
- **derived:** family identity, quantity, ledgers, B/F/K metrics, intervals, decisions.
- **generated:** chronology, customer/job identity, release/known/due times, priority, fallback layouts, LOCo candidates, stock boundaries.
- **assumed:** material identity and all economics.

**Permitted claim:** current modeled YieldForge algorithms did not demonstrate enough reliable economic value on either tested segment to justify continued investment, a pilot, or productization.

**Excluded claims:** not proof that no future algorithm/product form can work; not buyer willingness-to-pay evidence; not a live-factory causal trial.
